# L90 · 对话式 RAG — 会话记忆（Conversation Memory）、来源归因与 Podcast Q&A 流水线

**目标**：串联 RAG 检索 + 本地 LLM，构建能回答音频/播客内容问题的对话引擎

🔗 **Aurora 连接**：本节是 `aurora/podcast_intelligence/` 的完整交付物——`podcast_qa()` 函数将作为对话引擎的核心入口，供 L93/L94 的评估流水线和最终 demo 直接调用。

## 学习目标

1. 实现多轮对话记忆（滑动窗口 `truncate_history`）
2. 构建带来源归因的 RAG prompt（`build_prompt` + `format_sources`）
3. 完成 `podcast_qa()` 端到端检索生成流水线


## 核心直觉

RAG（Retrieval-Augmented Generation）的本质是把"记忆检索"和"语言生成"解耦：LLM 不需要把所有播客内容烧进权重，而是在回答时现查现用。检索层从向量索引（Vector Index）里捞出最相关的 3 个文本块（chunks），拼进 prompt 的 context 窗口；LLM 只读这些上下文生成答案，避免幻觉（Hallucination）。加上来源归因和滑动窗口（Sliding Window）历史，系统就具备了可追溯、可持续的对话能力。

In [ ]:
# ── aurora.llm 导入（TF-IDF 稀疏检索） ──────────────────────────────────────
#
# L89 实现了 build_tfidf / cosine_retrieve（词频统计 × 逆文档频率 的稀疏向量）。
# 本节在此基础上提供兼容接口 build_rag_index / retrieve，并演示如何扩展到
# 稠密嵌入（Dense Embedding）路径——见下方桩实现中的 embed_fn 参数。
#
# 稀疏 vs 稠密检索对比：
#   TF-IDF（L89/L90 aurora.llm）：词频统计，无需模型，速度快，OOV 词无效
#   Dense Embedding（L90 桩）：语义相似，需要嵌入模型，泛化更好但更重
try:
    from aurora.llm import build_tfidf, cosine_retrieve

    def build_rag_index(chunks, embed_fn=None):
        """使用 aurora.llm TF-IDF 构建检索索引。
        
        返回 (tfidf_matrix, vocab, chunks) 三元组供 retrieve() 使用。
        embed_fn 参数保留但在 TF-IDF 路径中忽略。
        """
        texts = [c["text"] if isinstance(c, dict) else c for c in chunks]
        matrix, vocab = build_tfidf(texts)
        return (matrix, vocab, chunks)

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """TF-IDF 余弦相似度检索，返回 top_k 个最相关的 chunk 字典列表。"""
        matrix, vocab, orig_chunks = rag_index
        texts = [c["text"] if isinstance(c, dict) else c for c in orig_chunks]
        results = cosine_retrieve(query, matrix, vocab, texts, top_k=top_k)
        text_set = {doc: orig_chunks[i] for i, (doc, _) in enumerate(
            zip(texts, [0]*len(texts)))}  # placeholder
        # Map retrieved text back to chunk dicts
        text_to_chunk = {}
        for ch in orig_chunks:
            t = ch["text"] if isinstance(ch, dict) else ch
            text_to_chunk[t] = ch
        return [text_to_chunk.get(doc, {"text": doc, "source": ""})
                for doc, _ in results]

    print("✅ aurora.llm 导入成功（TF-IDF 检索路径）")
except ImportError:
    import numpy as np

    def _cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    def build_rag_index(chunks, embed_fn=None):
        """稠密嵌入桩索引（aurora.llm 不可用时的后备）。
        
        返回嵌入矩阵 ndarray，形状 (n_chunks, dim)。
        注意：embed_fn 若为 None，则使用随机向量（仅供演示）。
        """
        import numpy as np
        if embed_fn is None:
            rng = np.random.default_rng(0)
            return rng.standard_normal((len(chunks), 8)).astype(np.float32)
        vecs = np.array([
            embed_fn(c["text"] if isinstance(c, dict) else c) for c in chunks
        ], dtype=np.float32)
        return vecs

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """稠密嵌入余弦检索（桩版本）。"""
        import numpy as np
        vecs = rag_index
        if embed_fn is None:
            return chunks[:top_k]
        q_vec = embed_fn(query)
        scores = [_cosine(q_vec, v) for v in vecs]
        idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [chunks[i] for i in idxs]

    print("⚠️  使用稠密嵌入桩（aurora.llm 未找到，请运行 make install）")


## 1. RAG Prompt 格式

RAG prompt 由四层拼接而成，顺序不能乱：

```
[SYSTEM]
你是播客助手，只基于提供的上下文回答问题，不要编造信息。

[CONTEXT]
--- 片段 1 (来源: episode_42.txt, t=00:12:30) ---
"John played lead guitar on that track..."

--- 片段 2 (来源: episode_42.txt, t=00:15:02) ---
"The band lineup included Sarah on bass..."

[HISTORY]
User: Who produced the album?
Assistant: The album was produced by Rick Rubin (来源: episode_42.txt, t=00:08:10).

[QUESTION]
Who played guitar in this episode?

仅基于以上内容回答。若上下文不含答案，请明确说"根据已有内容无法回答"。
```

关键设计决策：
- `[CONTEXT]` 在 `[HISTORY]` 之前——让模型先读事实再读对话流，减少历史对检索结果的干扰
- "仅基于以上内容回答" 是硬约束，防止模型混入预训练记忆
- 每个片段标注来源，方便后续来源归因

In [ ]:
def build_prompt(context_chunks, history, question, system_prompt=None):
    """
    把检索片段 + 对话历史 + 当前问题拼成完整 prompt 字符串。

    Parameters
    ----------
    context_chunks : list[dict]
        每个元素含 'text' 和 'source' 键。
    history : list[dict]
        每个元素含 'role'('user'/'assistant') 和 'content' 键。
    question : str
        当前用户问题。
    system_prompt : str | None
        若为 None 则使用默认系统提示。

    Returns
    -------
    str
        组装好的 prompt。
    """
    if system_prompt is None:
        system_prompt = (
            "你是播客助手，只基于提供的上下文回答问题，不要编造信息。\n"
            "若上下文不含答案，请明确说「根据已有内容无法回答」。"
        )

    parts = [f"[SYSTEM]\n{system_prompt}\n"]

    # 上下文片段
    parts.append("[CONTEXT]")
    for i, chunk in enumerate(context_chunks, 1):
        src = chunk.get("source", "未知来源")
        text = chunk.get("text", "")
        parts.append(f"--- 片段 {i} (来源: {src}) ---\n{text}")
    parts.append("")

    # 历史对话
    if history:
        parts.append("[HISTORY]")
        for turn in history:
            role = "User" if turn["role"] == "user" else "Assistant"
            parts.append(f"{role}: {turn['content']}")
        parts.append("")

    # 当前问题
    parts.append(f"[QUESTION]\n{question}\n\n仅基于以上内容回答。")

    return "\n".join(parts)


# 演示
demo_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
]
demo_history = [{"role": "user", "content": "Who produced the album?"},
                {"role": "assistant", "content": "Rick Rubin (来源: ep42.txt t=00:08:10)."}]
prompt = build_prompt(demo_chunks, demo_history, "Who played guitar?")
print(prompt)

## 2. 来源归因（Source Attribution）

每次检索都应该记录是从哪个文档的哪个位置找到的。格式：

```
来源：episode_42.txt, t=00:12:30
```

实现方式：`retrieve()` 返回的每个 chunk 字典里带 `source` 字段；`podcast_qa()` 把这些 source 列表一并返回给调用方，前端/评估脚本可以直接展示或记录日志。

来源归因的价值：
- 用户可验证答案的真实性（点击时间戳跳回播客）
- 评估时可自动对比"检索来源"与"标注答案来源"，计算 Precision@k

In [ ]:
def format_sources(source_chunks):
    """
    把来源 chunk 列表格式化为可读字符串列表。

    Parameters
    ----------
    source_chunks : list[dict]
        含 'source' 键的 chunk 列表。

    Returns
    -------
    list[str]
        格式化的来源字符串列表，去重并保序。
    """
    seen = set()
    result = []
    for chunk in source_chunks:
        src = chunk.get("source", "未知来源")
        if src not in seen:
            seen.add(src)
            result.append(f"📄 {src}")
    return result


# 演示
srcs = format_sources(demo_chunks)
for s in srcs:
    print(s)
print("✅ 来源格式化正常")

## 3. 会话记忆与滑动窗口

对话历史越长，token 消耗越多，最终会超出 LLM 的 context 长度限制。解决方案：**滑动窗口**——只保留最近 `k` 轮对话。

```
history = history[-(2*k):]   # 每轮含 user + assistant 两条，共 2k 条
```

更精细的做法是按 token 数截断而非轮数，但对播客 QA 场景，k=5 轮（≈1000 token）已足够。

注意：截断历史不影响向量检索的质量——检索走的是嵌入（Embedding）空间，与 history 无关。

In [ ]:
def truncate_history(history, max_turns=5):
    """
    保留最近 max_turns 轮对话（每轮 = user + assistant 各一条）。

    Parameters
    ----------
    history : list[dict]
        对话历史，每条含 'role' 和 'content'。
    max_turns : int
        最多保留的轮数。

    Returns
    -------
    list[dict]
        截断后的历史。
    """
    max_entries = max_turns * 2   # user + assistant
    return history[-max_entries:] if len(history) > max_entries else history


# 演示：构造 8 轮历史，截断到 3 轮
long_history = []
for i in range(8):
    long_history.append({"role": "user", "content": f"问题{i+1}"})
    long_history.append({"role": "assistant", "content": f"回答{i+1}"})

short = truncate_history(long_history, max_turns=3)
print(f"原始历史条数: {len(long_history)}")
print(f"截断后条数:   {len(short)}")
print(f"最早保留轮次: {short[0]['content']}")
print("✅ 滑动窗口正常")

## 4. ✏️ 实现 `podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None)`

**推理路线**：
1. `truncate_history(history, max_turns=5)` → 裁剪历史，防止 token 爆炸
2. `retrieve(question, rag_index, chunks, embed_fn, top_k=3)` → 拿到最相关的 3 个 context chunk
3. `build_prompt(context_chunks, history, question)` → 组装完整 prompt
4. `llm.generate(prompt)` → 调用本地 LLM 生成答案字符串
5. 返回 `(answer: str, sources: list[str])`，sources 由 `format_sources(context_chunks)` 生成

**参考输入输出**：

```python
question  = "Who played guitar in this episode?"
# retrieve 找到:
#   chunk_0: {"text": "John played lead guitar...", "source": "ep42.txt t=00:12:30"}
#   chunk_1: {"text": "The session featured...",   "source": "ep42.txt t=00:14:55"}
#   chunk_2: {"text": "Sarah on bass...",           "source": "ep42.txt t=00:15:02"}
# llm 生成:
answer  = "John played lead guitar in this episode."
sources = ["📄 ep42.txt t=00:12:30", "📄 ep42.txt t=00:14:55", "📄 ep42.txt t=00:15:02"]
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None):
    """
    播客问答主函数：检索相关片段 → 组装 prompt → LLM 生成 → 返回答案与来源。

    Parameters
    ----------
    question  : str          当前用户问题
    rag_index : np.ndarray   由 build_rag_index 生成的嵌入矩阵
    chunks    : list[dict]   原始 chunk 列表，每条含 'text' 和 'source'
    embed_fn  : callable     文本 → 向量的嵌入函数
    llm       : object       带 .generate(prompt: str) -> str 方法的 LLM 对象
    history   : list[dict]   对话历史，默认 None（空列表）

    Returns
    -------
    answer  : str
    sources : list[str]
    """
    if history is None:
        history = []

    # ✏️ TODO 步骤 1：用 truncate_history 裁剪 history（max_turns=5）
    history = ...

    # ✏️ TODO 步骤 2：调用 retrieve 获取 context_chunks（top_k=3）
    context_chunks = ...

    # ✏️ TODO 步骤 3：调用 build_prompt 组装完整 prompt
    prompt = ...

    # ✏️ TODO 步骤 4：调用 llm.generate(prompt) 得到 answer
    answer = ...

    # ✏️ TODO 步骤 5：调用 format_sources 并返回 (answer, sources)
    sources = ...
    return answer, sources

In [ ]:
# ── 最小桩 LLM，用于单元测试（不调用真实模型）──────────────────────────
import numpy as np

class _StubLLM:
    """返回与检索到的 context 相关答案的假 LLM，仅供检查用。"""
    def generate(self, prompt):
        # 从 prompt 中提取第一个 chunk 文本行作为答案基础
        lines = prompt.split("\n")
        for line in lines:
            line = line.strip()
            # chunk 文本行以引号包裹或直接出现在 [CONTEXT] 后
            if line and not line.startswith("[") and not line.startswith("---") and not line.startswith("你是") and len(line) > 10:
                return line  # 直接回显第一条 context 内容
        return "根据提供的上下文无法确定。"

def _stub_embed(text_or_chunk):
    """把文本哈希成 8 维向量（仅供测试）。"""
    text = text_or_chunk["text"] if isinstance(text_or_chunk, dict) else text_or_chunk
    rng = np.random.default_rng(abs(hash(text)) % (2**31))
    return rng.standard_normal(8).astype(np.float32)

# 构造测试数据
_test_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
    {"text": "The producer was Rick Rubin.", "source": "ep42.txt t=00:08:10"},
    {"text": "Drums were handled by Mike.", "source": "ep42.txt t=00:20:00"},
]
_test_index = build_rag_index(_test_chunks, _stub_embed)
_test_llm   = _StubLLM()

answer, sources = podcast_qa(
    question   = "Who played guitar in this episode?",
    rag_index  = _test_index,
    chunks     = _test_chunks,
    embed_fn   = _stub_embed,
    llm        = _test_llm,
    history    = [],
)

if answer is ... or answer is None:
    print('⬜ 请先实现 podcast_qa() 的5个TODO步骤')
else:
    print('答案：', answer)
    print('来源：')
    for s in sources:
        print('  ', s)
    assert isinstance(answer, str) and len(answer) > 0, 'answer 应为非空字符串'
    assert isinstance(sources, list) and len(sources) > 0, 'sources 应为非空列表'
    assert all(s.startswith('📄') for s in sources), '每条来源应以 📄 开头'
    print('\n✅ podcast_qa 基本检查通过')


## 5. 参数实验：有无 RAG 的回答质量对比

**实验设置**：
- 构建 10 个模拟播客 episode 的索引（每个 episode 4 个 chunk，共 40 条）
- 测试 5 个问题，分别记录：
  - `with_rag=True`：走完整 `podcast_qa()` 流程
  - `with_rag=False`：直接把问题喂给 LLM（不检索，不注入 context）

**预期现象**：
- `top_k=3`：通常足够，增大到 5 可能引入噪声 chunk，导致答案不聚焦
- `max_turns=5`：历史超过 5 轮时截断，答案稳定；设为 0 则每轮独立，失去上下文连贯性
- 无 RAG 时：LLM 对播客特定细节（人名、时间戳、观点）往往幻觉或回答"不知道"
- 有 RAG 时：答案直接引用 chunk 文本，来源可追溯

**关键指标**：`Precision@3`（3 个检索结果中有多少含正确答案的 chunk）

In [ ]:
import numpy as np

# ── 构造 10 个 episode × 4 chunk 的合成数据集 ──────────────────────
np.random.seed(42)

episode_facts = [
    ("Alice", "piano"), ("Bob", "guitar"), ("Carol", "drums"),
    ("Dave", "bass"),   ("Eve", "violin"), ("Frank", "trumpet"),
    ("Grace", "flute"), ("Hank", "cello"), ("Iris", "harp"), ("Jack", "sitar"),
]

synth_chunks = []
for ep_idx, (name, instrument) in enumerate(episode_facts):
    ep_name = f"episode_{ep_idx+1:02d}.txt"
    synth_chunks += [
        {"text": f"{name} played {instrument} in the session.",
         "source": f"{ep_name} t=00:10:{ep_idx*3:02d}"},
        {"text": f"The host interviewed {name} about their career.",
         "source": f"{ep_name} t=00:20:{ep_idx*2:02d}"},
        {"text": f"Audience Q&A focused on {instrument} technique.",
         "source": f"{ep_name} t=00:35:00"},
        {"text": f"Closing remarks from {name}.",
         "source": f"{ep_name} t=00:55:00"},
    ]

synth_index = build_rag_index(synth_chunks, _stub_embed)

# ── 5 个测试问题（附正确答案供对比）──────────────────────────────────
test_questions = [
    ("Who plays guitar?",  "Bob"),
    ("Who plays piano?",   "Alice"),
    ("Who plays drums?",   "Carol"),
    ("Who plays violin?",  "Eve"),
    ("Who plays cello?",   "Hank"),
]

# ── 对比实验 ───────────────────────────────────────────────────────────
class _NoContextLLM:
    """模拟无 RAG 时 LLM 的行为：不看上下文，直接猜。"""
    def generate(self, prompt):
        return "我不确定，可能是某位音乐家。"   # 典型幻觉/拒答

print(f"{'问题':<30} {'期望':<10} {'RAG答案':<45} {'无RAG答案'}")
print("-" * 110)

for question, expected in test_questions:
    # 有 RAG
    ans_rag, srcs = podcast_qa(question, synth_index, synth_chunks, _stub_embed, _test_llm)
    if ans_rag is ... or ans_rag is None:
        print("⬜ 请先实现 podcast_qa() 再运行对比实验")
        break
    # 无 RAG（直接调用 LLM，跳过检索）
    ans_nrag = _NoContextLLM().generate(question)
    hit = "✅" if expected.lower() in ans_rag.lower() else "❌"
    print(f"{question:<30} {expected:<10} {hit} {ans_rag:<43} {ans_nrag}")

print()
print("观察：有 RAG 时答案直接引用 chunk 文本；无 RAG 时 LLM 无法回答播客特定细节。")

## 本课收束

本节实现了 `podcast_qa(question, rag_index, chunks, embed_fn, llm, history)` —— 它将检索到的 context chunks 拼入结构化 prompt，输出 `(answer: str, sources: list[str])` 两元组，让每条答案都可追溯到具体播客时间戳。`build_prompt`、`truncate_history`、`format_sources` 三个辅助函数共同构成 `aurora/podcast_intelligence/qa.py` 的完整骨架。下一课：**L91**。